In [ ]:
# This script wil initialize databases and tables

In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from lib.aws_s3 import warehouse_dir

In [ ]:
# Create the SparkSession
"""
spark = (
    SparkSession.builder
    .appName("01 - Initialize database")
    .config("spark.sql.warehouse.dir", warehouse_dir)
    .enableHiveSupport()
    .getOrCreate()
)
spark
"""

In [ ]:
print("****************************************")
print("01 - Initialize database")
print("****************************************")

In [ ]:
# Dropping tables and databases if they exist
RESET_ENV = True
db_names = ["lnd", "stg", "prd"]

failed = []

if RESET_ENV:
    for db in db_names:
        try:
            tables = spark.catalog.listTables(db)
        except Exception as e:
            #print(f"Skipping database {db}: {e}")
            continue

        for t in tables:
            if t.isTemporary:
                continue
            kind = "VIEW" if t.tableType == "VIEW" else "TABLE"
            try:
                spark.sql(f"DROP {kind} IF EXISTS {db}.{t.name}")
                #print(f"Dropped {db}.{t.name}")
            except Exception as e:
                failed.append((f"{db}.{t.name}", str(e)[:200]))

    if failed:
        print("\nCould not drop:")
        for name, err in failed:
            #print(f"  {name}: {err}")
            pass

In [ ]:
# Create dw schema in catalog
print("Creating databases: lnd, stg, prd\n")

# Landing
spark.sql("create database if not exists lnd");

# Staging
spark.sql("create database if not exists stg");

# Production
spark.sql("create database if not exists prd");

# Show all databases
spark.sql("show databases").show();

In [ ]:
# Date table
spark.sql("drop table if exists prd.dim_date")
spark.sql("""
create table prd.dim_date (
    date date,
    day int,
    month int,
    year int,
    quarter int,
    day_of_week string,
    is_weekend boolean,
    insert_ts timestamp
) using delta
""")
print("prd.dim_date table created")

In [ ]:
spark.table('prd.dim_date').printSchema()

In [ ]:
# Dealer table
spark.sql("drop table if exists prd.dim_dealer")
spark.sql("""
create table prd.dim_dealer (
    dealer_key bigint,
    dealer_id string,
    dealer_name string,
    oem_name string,
    address string,
    city string,
    state string,
    zipcode string,
    phone_number string,
    is_current boolean,
    effective_start_date date,
    effective_end_date date,
    insert_ts timestamp,
    update_ts timestamp
) using delta
""")
print("prd.dim_dealer table created")

In [ ]:
spark.table('prd.dim_dealer').printSchema()

In [ ]:
# Customer table
spark.sql("drop table if exists prd.dim_customer")
spark.sql("""
create table prd.dim_customer (
    customer_key bigint,
    customer_id string,
    first_name string,
    last_name string,
    address string,
    city string,
    state string,
    zipcode string,
    email string,
    phone_number string,
    customer_type string,
    is_current boolean,
    effective_start_date date,
    effective_end_date date,
    is_inferred boolean,
    insert_ts timestamp,
    update_ts timestamp
) using delta
""")
print("prd.dim_customer table created")

In [ ]:
spark.table('prd.dim_customer').printSchema()

In [ ]:
# Operation table
spark.sql("drop table if exists prd.dim_operation")
spark.sql("""
create table prd.dim_operation (
    operation_key bigint,
    operation_code string,
    operation_name string,
    operation_category string,
    insert_ts timestamp,
    update_ts timestamp
) using delta
""")
print("prd.dim_operation table created")

In [ ]:
spark.table('prd.dim_operation').printSchema()

In [ ]:
# Part table
spark.sql("drop table if exists prd.dim_part")
spark.sql("""
create table prd.dim_part (
    part_key bigint,
    part_code string,
    part_name string,
    part_category string,
    insert_ts timestamp,
    update_ts timestamp
) using delta
""")
print("prd.dim_part table created")

In [ ]:
spark.table('prd.dim_part').printSchema()

In [ ]:
# Vehicle table
spark.sql("drop table if exists prd.dim_vehicle")
spark.sql("""
create table prd.dim_vehicle (
    vin string,
    make string,
    model string,
    model_year int,
    trim string,
    fuel_type string,
    drive_type string,
    insert_ts timestamp
) using delta
""")
print("prd.dim_vehicle table created")

In [ ]:
spark.table('prd.dim_vehicle').printSchema()

In [ ]:
# Service table
spark.sql("drop table if exists prd.fact_service")
spark.sql("""
create table prd.fact_service (
    ro_number bigint,
    line_number bigint,
    dealer_key bigint,
    customer_key bigint,
    vin string,
    service_open_date date,
    service_close_date date,
    odometer bigint,
    operation_key bigint,
    part_key bigint,
    line_type string,
    pay_type string,
    quantity double,
    unit_price double,
    line_amount double,
    insert_ts timestamp
) using delta
""")
print("prd.fact_service table created")

In [ ]:
spark.table('prd.fact_service').printSchema()

In [ ]:
# Job Control table
spark.sql("drop table if exists prd.job_control")
spark.sql("""
create table prd.job_control (
    schema_name string,
    table_name string,
    insert_ts timestamp,
    run_date date
) using delta
""")
print("prd.job_control table created")

In [ ]:
spark.table('prd.job_control').printSchema()

In [ ]:
spark.sql("show tables in prd").show();

In [ ]:
# Stop spark session
#spark.stop()